
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 05: Pandas II — Data Wrangling

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Transformar, combinar y limpiar datasets reales usando las operaciones de Pandas más usadas en un flujo de trabajo profesional.

## 🗺️ Tabla de Contenido
1. [Introducción](#intro)
2. [groupby y agregaciones](#groupby)
3. [Combinar datasets: merge y concat](#merge)
4. [Tablas dinámicas: pivot_table](#pivot)
5. [Valores faltantes (introducción)](#faltantes)
6. [Columnas derivadas: apply, map y fechas](#derivadas)
7. [Ejemplos de aplicación real](#aplicaciones)
8. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

"Data Wrangling" es transformar datos crudos y desordenados en algo listo para analizar. Es el equivalente en Pandas a las tablas dinámicas, `BUSCARV` y filtros de Excel — pero con código, reproducible y escalable a millones de filas.

In [ ]:
import pandas as pd
import numpy as np

ventas = pd.DataFrame({
    "fecha": pd.to_datetime([
        "2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08",
        "2026-01-15", "2026-01-16", "2026-02-02", "2026-02-03",
    ]),
    "region": ["Norte", "Sur", "Norte", "Centro", "Sur", "Norte", "Centro", "Sur"],
    "categoria": ["Electro", "Hogar", "Hogar", "Electro", "Electro", "Hogar", "Hogar", "Electro"],
    "vendedor_id": [1, 2, 1, 3, 2, 1, 3, 2],
    "ingresos": [1500000, 980000, 2100000, 1230000, 870000, 1650000, np.nan, 1990000],
})
ventas

<a id="groupby"></a>
## 2. groupby y Agregaciones

### 🔬 Teoría técnica
`groupby` divide el DataFrame en grupos según una o más columnas, y luego aplica una función de agregación (`sum`, `mean`, `count`, etc.) a cada grupo — el mismo concepto que "resumir por categoría" que ya practicaste con diccionarios en la Sesión 03, pero automático y vectorizado.

In [ ]:
# Ingresos totales por región
ventas.groupby("region")["ingresos"].sum()

In [ ]:
# Múltiples agregaciones a la vez con .agg()
ventas.groupby("region")["ingresos"].agg(["sum", "mean", "count"])

In [ ]:
# Agrupar por más de una columna
ventas.groupby(["region", "categoria"])["ingresos"].sum().reset_index()

### 🧠 Resumen para dummies
`groupby("columna")["otra_columna"].sum()` se lee como: "agrupa por columna, y de cada grupo súmame otra_columna".

<a id="merge"></a>
## 3. Combinar Datasets: merge y concat

### 🔬 Teoría técnica
- **`pd.merge`**: cruza dos tablas por una columna en común (como `BUSCARV`/`JOIN` en SQL). Tipos: `inner` (solo coincidencias), `left`/`right` (todo de una tabla + coincidencias), `outer` (todo de ambas).
- **`pd.concat`**: apila DataFrames uno encima de otro (mismas columnas) o uno al lado del otro.

In [ ]:
vendedores = pd.DataFrame({
    "vendedor_id": [1, 2, 3],
    "nombre_vendedor": ["Camila", "Jorge", "Valentina"],
})

# merge tipo "left": conservamos todas las ventas, aunque falte info del vendedor
ventas_completas = ventas.merge(vendedores, on="vendedor_id", how="left")
ventas_completas

In [ ]:
# concat: apilar dos DataFrames con las mismas columnas
ventas_marzo = pd.DataFrame({
    "fecha": pd.to_datetime(["2026-03-01", "2026-03-02"]),
    "region": ["Norte", "Sur"],
    "categoria": ["Hogar", "Electro"],
    "vendedor_id": [1, 2],
    "ingresos": [1750000, 2050000],
})

ventas_totales = pd.concat([ventas, ventas_marzo], ignore_index=True)
print(ventas_totales.shape)
ventas_totales.tail()

### 💪 Fortalezas y debilidades
- `merge` con `how="inner"` puede **perder filas** silenciosamente si no hay coincidencia — siempre revisa `.shape` antes y después.
- `concat` requiere que las columnas coincidan (o tendrás columnas con `NaN` donde no hay dato).

### 🧠 Resumen para dummies
`merge` = "cruzar dos tablas por una clave en común". `concat` = "pegar tablas una debajo/al lado de otra".

<a id="pivot"></a>
## 4. Tablas Dinámicas: pivot_table

### 🔬 Teoría técnica
`pivot_table` reorganiza los datos: convierte valores únicos de una columna en **nuevas columnas**, cruzándolos contra otra variable — exactamente como una tabla dinámica de Excel.

In [ ]:
tabla_dinamica = ventas.pivot_table(
    values="ingresos",
    index="region",
    columns="categoria",
    aggfunc="sum",
    fill_value=0,
)
tabla_dinamica

### 🧠 Resumen para dummies
`index` = filas del resultado, `columns` = columnas del resultado, `values` = qué se calcula, `aggfunc` = cómo se calcula (suma, promedio, etc.).

<a id="faltantes"></a>
## 5. Valores Faltantes (Introducción)

### 🔬 Teoría técnica
`isna()` detecta valores nulos (`NaN`), `dropna()` los elimina, `fillna()` los reemplaza. Aquí solo se introduce la sintaxis; **las estrategias correctas de imputación** (cuándo usar media, mediana, moda, etc.) se estudian a fondo en la Sesión 07.

In [ ]:
print(ventas.isna().sum())  # cuántos nulos hay por columna

# Reemplazar temporalmente con 0 solo para poder sumar sin errores
ventas_sin_nulos = ventas.fillna({"ingresos": 0})
ventas_sin_nulos

<a id="derivadas"></a>
## 6. Columnas Derivadas: apply, map y Fechas

### 🔬 Teoría técnica
- **`.map()`**: transforma **elemento por elemento** una Serie (una sola columna).
- **`.apply()`**: aplica una función a cada fila o columna completa (más flexible, puede usar varias columnas a la vez).
- Para trabajar con fechas, primero se convierten con `pd.to_datetime()` (ya lo hicimos al crear `ventas`), y luego se accede a sus componentes con el "acceso" `.dt`.

In [ ]:
# map: transformar una sola columna
region_map = {"Norte": "N", "Sur": "S", "Centro": "C"}
ventas["region_abrev"] = ventas["region"].map(region_map)

# apply sobre una fila completa: crear una columna combinando 2+ columnas
ventas["etiqueta"] = ventas.apply(lambda fila: f"{fila['region']}-{fila['categoria']}", axis=1)

# Extraer componentes de fecha
ventas["mes"] = ventas["fecha"].dt.month
ventas["dia_semana"] = ventas["fecha"].dt.day_name()

ventas[["fecha", "region_abrev", "etiqueta", "mes", "dia_semana"]]

### 🧠 Resumen para dummies
`map` es para "cambiar cada valor de una columna por otro" (como una tabla de traducción). `apply` es para "calcular algo nuevo mirando varias columnas de la misma fila". `.dt.mes`, `.dt.dia_semana`, etc. sacan pedazos útiles de una fecha.

<a id="aplicaciones"></a>
## 7. Ejemplos de Aplicación en el Mundo Real

- Consolidar reportes de ventas por región y mes para una gerencia comercial.
- Cruzar (`merge`) una tabla de clientes con una de transacciones para calcular el valor de vida de cada cliente.
- Preparar variables de fecha (`mes`, `día de la semana`) como *features* para un modelo de series de tiempo o de demanda.

<a id="retos"></a>
## 8. Retos de Práctica

### 🥉 Reto Básico
Sobre `ventas`, calcula el ingreso promedio agrupado por `categoria` usando `groupby`.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Crea una tabla dinámica (`pivot_table`) que muestre el ingreso promedio cruzando `region` (filas) y `mes` (columnas). Luego, une (`merge`) el resultado de `ventas_completas` con una tabla adicional que tú crees con una "meta de ventas" por vendedor, y calcula el cumplimiento (ingresos reales / meta).

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Construye un mini-pipeline de wrangling: parte de `ventas` y `vendedores`, únelas con `merge`, crea al menos 2 columnas derivadas (una con `apply`, otra extrayendo un componente de fecha con `.dt`), maneja los valores faltantes de `ingresos` con `fillna` usando el promedio de su región (no un valor fijo), y produce un resumen final agregado con `.agg()` usando al menos 2 funciones distintas por columna.

In [ ]:
# Tu solución al Reto Avanzado aquí
